# GPU Backend Example

This example shows how to run `mdopt` on a GPU through [CuPy](https://cupy.dev/),
and compares the cost of the same MPS-MPO contraction on CPU and GPU.

`mdopt` picks its array backend from the `MDOPT_BACKEND` environment variable,
and it does so **once, when `mdopt.backend.array` is first imported**. A single
Python process therefore cannot switch backends after the fact. To keep this
notebook runnable top to bottom, each benchmark is launched in its own
subprocess with the appropriate `MDOPT_BACKEND` value, so no kernel restarts
are needed and both results end up in one run.

To run this on Colab, first switch the runtime to a GPU:
**Runtime → Change runtime type → Hardware accelerator → GPU**.


## Setup

Install the packages and make sure numpy is in a consistent state.


In [ ]:
# Install mdopt. Colab's GPU runtimes already ship CuPy, pinned to a version
# whose bundled CUDA runtime matches the driver on the VM. Reinstalling it pulls
# the latest wheel, which can need a newer driver than the VM has and then fails
# with "cudaErrorInsufficientDriver" once a CUDA call is made -- so we
# deliberately leave CuPy alone here.
!pip -q install "git+https://github.com/quicophy/mdopt.git"

# mdopt requires numpy>=2.3, so pip upgrades the numpy that Colab preloads at
# startup. Because the old numpy is already imported by the running kernel, the
# upgrade can leave a mix of new Python files and the old compiled extension,
# which shows up as "ImportError: cannot import name '_slice' from
# numpy._core.umath". Reinstalling numpy once, cleanly, repairs that.
!pip -q install --force-reinstall --no-cache-dir "numpy>=2.3,<2.5"

# Report the environment from *fresh* processes: this kernel may still hold the
# stale numpy, but the benchmarks below run in subprocesses and so pick up the
# repaired one. The driver and CUDA runtime versions are printed together, since
# a GPU run needs the driver to be new enough for the runtime CuPy was built for.
!python -c "import numpy; print('numpy', numpy.__version__, 'imports cleanly')"
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader || echo "no GPU on this runtime"
!python -c "import cupy; print('cupy', cupy.__version__, '| CUDA runtime', cupy.cuda.runtime.runtimeGetVersion())" || echo "CuPy unavailable"


Write the benchmark to a file so both runs execute exactly the same code.


In [ ]:
%%writefile bench.py
"""MPS-MPO contraction benchmark. The backend is chosen by MDOPT_BACKEND."""

import os
import time

from tqdm import tqdm

from mdopt.mps.utils import create_simple_product_state
from mdopt.utils.utils import create_random_mpo
from mdopt.contractor.contractor import mps_mpo_contract
from mdopt.backend import array as A

LABEL = os.environ.get("BENCH_LABEL", A.backend_name().upper())


def bench(num_sites=48, phys_dim=2, mpo_len=32, chi=256, reps=10):
    """Time one MPS-MPO contraction, averaged over `reps` repetitions."""
    mps = create_simple_product_state(
        num_sites=num_sites, which="0", phys_dim=phys_dim
    )
    mpo = create_random_mpo(
        num_sites=mpo_len,
        bond_dimensions=[chi] * (mpo_len - 1),
        phys_dim=phys_dim,
        which="uniform",
    )

    # Warm-up, so one-off allocation and CUDA kernel compilation are not timed.
    mps_mpo_contract(mps.copy(), mpo, start_site=0, renormalise=False)
    A.synchronize()

    t0 = time.perf_counter()
    for _ in tqdm(range(reps)):
        mps_mpo_contract(mps.copy(), mpo, start_site=0, renormalise=False)
    A.synchronize()
    t1 = time.perf_counter()

    seconds = (t1 - t0) / reps
    print(f"{LABEL}: {seconds:.4f} s per run")
    return seconds


if __name__ == "__main__":
    print(f"Backend: {A.backend_name()} | GPU flag: {A.GPU}")
    bench()


## CPU baseline

Run with the numpy backend.


In [ ]:
# `python` is the same interpreter the notebook runs on; the setup cell above
# already used it. Each benchmark runs as a separate process so that
# MDOPT_BACKEND is read fresh on import.
!MDOPT_BACKEND=numpy BENCH_LABEL=CPU python bench.py


## GPU run

Same code, same parameters, with the CuPy backend.
If `GPU flag` prints `False`, the runtime has no GPU and `mdopt` silently fell
back to numpy — switch the runtime to GPU and re-run this cell.


In [ ]:
!nvidia-smi

!MDOPT_BACKEND=cupy BENCH_LABEL=GPU python bench.py
